# NAIC WP7 UC3: Pseudo-Hamiltonian Neural Networks

**Physics-preserving neural networks for dynamical systems**

This notebook demonstrates Pseudo-Hamiltonian Neural Networks (PHNNs) — neural network architectures that respect energy conservation, dissipation, and external forcing by construction. Standard neural networks trained on physical systems drift into unphysical states over long horizons. PHNNs prevent this by decomposing dynamics into three sub-networks rooted in port-Hamiltonian theory.

**What you will learn:**
1. How to simulate a damped mass-spring system using `phlearn`
2. How to train a PHNN vs. a standard baseline neural network
3. Why structural constraints matter for long-horizon prediction
4. How to add and learn external forces
5. How PHNNs generalize to PDE systems (KdV equation)

**References:**
- Eidnes et al. (2023). *Pseudo-Hamiltonian neural networks for learning partial differential equations.* Journal of Computational Physics, 500, 112738.
- Eidnes and Lye (2024). *Pseudo-Hamiltonian neural networks with state-dependent external forces.* Applied Mathematics and Computation, 459, 128242.

---

## 0. Setup

If running on an NAIC Orchestrator VM, the environment should already be configured via `setup.sh`. If not, uncomment the install line below.

In [ ]:
# Uncomment if phlearn is not installed:
# !pip install -e ../phlearn

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

import phlearn.phsystems.ode as phsys
import phlearn.phnns as phnn

ttype = torch.float32
torch.set_default_dtype(ttype)

# Use GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device: {device}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. The Problem: Neural Networks Drift on Physical Systems

Consider a damped harmonic oscillator (mass on a spring with friction). The Hamiltonian (total energy) is:

$$H(q, p) = \frac{k}{2} q^2 + \frac{1}{2m} p^2$$

where $q$ is position, $p$ is momentum, $k$ is the spring constant, and $m$ is the mass.

With damping coefficient $\gamma$, the equations of motion are:

$$\frac{dx}{dt} = (S - R) \nabla H(x)$$

where $S$ is skew-symmetric (energy-conserving) and $R$ is positive semi-definite (dissipation).

A standard neural network has no concept of energy — it can predict trajectories that create energy from nothing. A PHNN encodes the Hamiltonian structure directly, so energy can only decrease (through dissipation) or remain constant.

## 2. Define the Ground-Truth System

We create a damped mass-spring system with known parameters. This serves as our ground truth for generating training data and evaluating learned models.

In [ ]:
nstates = 2  # (position q, momentum p)

# Physical parameters
mass = 1.0
spring_constant = 1.0
damping = 0.3

# Dissipation matrix: only momentum is damped
R = np.diag([0, damping])

# Hamiltonian: H(q,p) = k/2 * q^2 + 1/(2m) * p^2
M = np.diag([spring_constant / 2, 1 / (2 * mass)])

def hamiltonian(x):
    return x.T @ M @ x

def hamiltonian_grad(x):
    return 2 * M @ x

spring_system = phsys.PseudoHamiltonianSystem(
    nstates=nstates,
    hamiltonian=hamiltonian,
    grad_hamiltonian=hamiltonian_grad,
    dissipation_matrix=R,
)

print(f'System: damped mass-spring')
print(f'  mass = {mass}, k = {spring_constant}, damping = {damping}')
print(f'  States: position (q), momentum (p)')

## 3. Generate Training Data

We sample multiple trajectories from the ground-truth system with random initial conditions. These trajectories are numerical solutions to the exact ODE.

In [ ]:
# Training data parameters
data_points = 30000
dt = 0.1
tmax = 10

nt = round(tmax / dt)
t_axis = np.linspace(0, tmax, nt + 1)
ntrajectories = int(np.ceil(data_points / nt))

traindata = phnn.generate_dataset(spring_system, ntrajectories, t_axis)

print(f'Generated {ntrajectories} trajectories')
print(f'  Time steps per trajectory: {nt}')
print(f'  Total data points: ~{ntrajectories * nt}')
print(f'  dt = {dt}, tmax = {tmax}')

### Visualize training trajectories

Each trajectory starts from a different initial condition. In phase space (q vs p), damped oscillators spiral inward toward the origin as energy dissipates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot a few sample trajectories in phase space
ax = axes[0]
for i in range(min(5, ntrajectories)):
    x_traj, *_ = spring_system.sample_trajectory(t_axis)
    ax.plot(x_traj[:, 0], x_traj[:, 1], alpha=0.7)
    ax.plot(x_traj[0, 0], x_traj[0, 1], 'ko', markersize=4)
ax.set_xlabel('Position (q)')
ax.set_ylabel('Momentum (p)')
ax.set_title('Phase Space Trajectories')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Plot energy decay for one trajectory
ax = axes[1]
x_traj, *_ = spring_system.sample_trajectory(t_axis, x0=[1.0, 0.0])
energies = [hamiltonian(x_traj[i]) for i in range(len(t_axis))]
ax.plot(t_axis, energies, 'b-', linewidth=2)
ax.set_xlabel('Time')
ax.set_ylabel('Energy H(q, p)')
ax.set_title('Energy Decay (Damping = 0.3)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Train PHNN vs. Standard Baseline

We train two models on the same data:

1. **PHNN** — decomposes dynamics into Hamiltonian + dissipation sub-networks with structural constraints
2. **Baseline NN** — a standard MLP that learns the full right-hand side without any physics

Both models have similar parameter counts.

In [ ]:
# PHNN: learns Hamiltonian and dissipation separately
states_dampened = np.diagonal(spring_system.dissipation_matrix) != 0
phmodel = phnn.PseudoHamiltonianNN(
    nstates,
    dissipation_est=phnn.R_estimator(states_dampened),
)

# Baseline: standard MLP, no physics
baseline_nn = phnn.BaselineNN(nstates, hidden_dim=100)
basemodel = phnn.DynamicSystemNN(nstates, baseline_nn)

print('Models created:')
print(f'  PHNN:     Hamiltonian NN + R_estimator (physics-constrained)')
print(f'  Baseline: MLP with hidden_dim=100 (unconstrained)')

In [ ]:
# Train both models
epochs = 30
batch_size = 32

print(f'Training PHNN ({epochs} epochs)...')
phmodel, ph_losses = phnn.train(
    phmodel, integrator='midpoint', traindata=traindata,
    epochs=epochs, batch_size=batch_size,
)

print(f'\nTraining Baseline ({epochs} epochs)...')
basemodel, base_losses = phnn.train(
    basemodel, integrator='midpoint', traindata=traindata,
    epochs=epochs, batch_size=batch_size,
)

print('\nTraining complete.')

## 5. Compare: Short-Horizon Prediction

On the training time window (0 to 10), both models perform well. The difference appears when we look at the learned physical quantities.

In [ ]:
# Generate test trajectory
x0 = [1.0, 0.0]
x_exact, *_ = spring_system.sample_trajectory(t_axis, x0=x0)
x_phnn, _ = phmodel.simulate_trajectory(integrator=False, t_sample=t_axis, x0=x0)
x_baseline, _ = basemodel.simulate_trajectory(integrator=False, t_sample=t_axis, x0=x0)

# Check learned damping constant
learned_damping = phmodel.R().detach().numpy()[1, 1]
true_damping = spring_system.dissipation_matrix[1, 1]
print(f'Damping constant:')
print(f'  True:    {true_damping:.4f}')
print(f'  Learned: {learned_damping:.4f}')
print(f'  Error:   {abs(true_damping - learned_damping):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Phase portrait
ax = axes[0]
ax.plot(x_exact[:, 0], x_exact[:, 1], 'k--', linewidth=2, label='Exact')
ax.plot(x_phnn[:, 0], x_phnn[:, 1], 'b-', linewidth=1.5, label='PHNN')
ax.plot(x_baseline[:, 0], x_baseline[:, 1], 'r-', linewidth=1.5, label='Baseline')
ax.plot(x0[0], x0[1], 'ko', markersize=8)
ax.set_xlabel('Position (q)')
ax.set_ylabel('Momentum (p)')
ax.set_title('Phase Portrait (t = 0 to 10)')
ax.legend()
ax.grid(True, alpha=0.3)

# Position vs time
ax = axes[1]
ax.plot(t_axis, x_exact[:, 0], 'k--', linewidth=2, label='Exact')
ax.plot(t_axis, x_phnn[:, 0], 'b-', linewidth=1.5, label='PHNN')
ax.plot(t_axis, x_baseline[:, 0], 'r-', linewidth=1.5, label='Baseline')
ax.set_xlabel('Time')
ax.set_ylabel('Position (q)')
ax.set_title('Position vs Time')
ax.legend()
ax.grid(True, alpha=0.3)

# Momentum vs time
ax = axes[2]
ax.plot(t_axis, x_exact[:, 1], 'k--', linewidth=2, label='Exact')
ax.plot(t_axis, x_phnn[:, 1], 'b-', linewidth=1.5, label='PHNN')
ax.plot(t_axis, x_baseline[:, 1], 'r-', linewidth=1.5, label='Baseline')
ax.set_xlabel('Time')
ax.set_ylabel('Momentum (p)')
ax.set_title('Momentum vs Time')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. The Key Test: Long-Horizon Extrapolation

This is where physics-informed structure pays off. We integrate both models for **5x longer** than the training window (t = 0 to 50). The PHNN should maintain physically plausible trajectories because its architecture guarantees energy can only decrease. The baseline has no such guarantee.

In [ ]:
# Long-horizon prediction: 5x training window
t_long = np.linspace(0, 50, 501)
x0_long = [1.5, 0.5]  # different initial condition

x_exact_long, *_ = spring_system.sample_trajectory(t_long, x0=x0_long)
x_phnn_long, _ = phmodel.simulate_trajectory(integrator=False, t_sample=t_long, x0=x0_long)
x_baseline_long, _ = basemodel.simulate_trajectory(integrator=False, t_sample=t_long, x0=x0_long)

# Compute energy for each trajectory
energy_exact = [hamiltonian(x_exact_long[i]) for i in range(len(t_long))]
energy_phnn = [hamiltonian(x_phnn_long[i]) for i in range(len(t_long))]
energy_baseline = [hamiltonian(x_baseline_long[i]) for i in range(len(t_long))]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Phase portrait - long horizon
ax = axes[0]
ax.plot(x_exact_long[:, 0], x_exact_long[:, 1], 'k--', linewidth=2, label='Exact', alpha=0.7)
ax.plot(x_phnn_long[:, 0], x_phnn_long[:, 1], 'b-', linewidth=1.5, label='PHNN')
ax.plot(x_baseline_long[:, 0], x_baseline_long[:, 1], 'r-', linewidth=1.5, label='Baseline', alpha=0.7)
ax.plot(x0_long[0], x0_long[1], 'ko', markersize=8)
ax.set_xlabel('Position (q)')
ax.set_ylabel('Momentum (p)')
ax.set_title('Phase Portrait (t = 0 to 50)')
ax.legend()
ax.grid(True, alpha=0.3)

# Position vs time - long horizon
ax = axes[1]
ax.plot(t_long, x_exact_long[:, 0], 'k--', linewidth=2, label='Exact', alpha=0.7)
ax.plot(t_long, x_phnn_long[:, 0], 'b-', linewidth=1.5, label='PHNN')
ax.plot(t_long, x_baseline_long[:, 0], 'r-', linewidth=1.5, label='Baseline', alpha=0.7)
ax.axvline(x=10, color='gray', linestyle=':', alpha=0.5, label='Training horizon')
ax.set_xlabel('Time')
ax.set_ylabel('Position (q)')
ax.set_title('Position: Training vs Extrapolation')
ax.legend()
ax.grid(True, alpha=0.3)

# Energy comparison
ax = axes[2]
ax.plot(t_long, energy_exact, 'k--', linewidth=2, label='Exact', alpha=0.7)
ax.plot(t_long, energy_phnn, 'b-', linewidth=1.5, label='PHNN')
ax.plot(t_long, energy_baseline, 'r-', linewidth=1.5, label='Baseline', alpha=0.7)
ax.axvline(x=10, color='gray', linestyle=':', alpha=0.5, label='Training horizon')
ax.set_xlabel('Time')
ax.set_ylabel('Energy H(q, p)')
ax.set_title('Energy Over Time')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Quantitative comparison
mse_phnn = np.mean((x_exact_long - x_phnn_long)**2)
mse_baseline = np.mean((x_exact_long - x_baseline_long)**2)
print(f'\nLong-horizon MSE (t=0 to 50):')
print(f'  PHNN:     {mse_phnn:.6f}')
print(f'  Baseline: {mse_baseline:.6f}')

# Check energy behavior -- the real test
energy_final_exact = energy_exact[-1]
energy_final_phnn = energy_phnn[-1]
energy_final_baseline = energy_baseline[-1]
print(f'\nEnergy at t=50:')
print(f'  Exact:    {energy_final_exact:.6f}')
print(f'  PHNN:     {energy_final_phnn:.6f}')
print(f'  Baseline: {energy_final_baseline:.6f}')
print(f'\nPHNN energy monotonically decreases (physical).')
print(f'Baseline energy may grow or oscillate (unphysical).')

## 7. Learning External Forces

A key advantage of PHNNs: you can add an external force sub-network that learns state- or time-dependent forcing. After training, you can *remove or modify* the force without retraining the conservative and dissipative parts.

We add a sinusoidal driving force: $F(t) = 0.5 \sin(\pi t / 2)$, acting on momentum only.

In [ ]:
# Define the external force
# phlearn calls this with varying shapes: (1,2)/(1,1) during integration, (T,2)/(T,) for batch eval
def external_force(x, t):
    f0 = 0.5
    omega = np.pi / 2
    t_arr = np.asarray(t).flatten()
    force_val = f0 * np.sin(omega * t_arr)
    result = np.zeros_like(np.atleast_2d(x))
    result[..., 1] = force_val.flatten()[:result.shape[0]] if result.ndim > 1 else force_val
    return result.squeeze() if np.asarray(x).ndim < 2 else result

# Create forced system
forced_system = phsys.PseudoHamiltonianSystem(
    nstates=nstates,
    hamiltonian=hamiltonian,
    grad_hamiltonian=hamiltonian_grad,
    dissipation_matrix=R,
    external_forces=external_force,
)

# Generate training data from the forced system
traindata_forced = phnn.generate_dataset(
    forced_system, ntrajectories, t_axis
)

print('Forced system created: F(t) = 0.5 * sin(pi*t/2) on momentum')

In [ ]:
# Build a PHNN with an external force estimator
ext_forces_nn = phnn.ExternalForcesNN(
    nstates=nstates,
    noutputs=1,
    external_forces_filter=[0, 1],  # force only on momentum
    hidden_dim=100,
    timedependent=True,
    statedependent=False,
)

phmodel_forced = phnn.PseudoHamiltonianNN(
    nstates,
    dissipation_est=phnn.R_estimator(states_dampened),
    external_forces_est=ext_forces_nn,
)

# Baseline for comparison
basemodel_forced = phnn.DynamicSystemNN(
    nstates, phnn.BaselineNN(nstates, hidden_dim=100)
)

# Train both
print('Training PHNN with external forces...')
phmodel_forced, _ = phnn.train(
    phmodel_forced, integrator='midpoint', traindata=traindata_forced,
    epochs=30, batch_size=32,
)

print('\nTraining Baseline...')
basemodel_forced, _ = phnn.train(
    basemodel_forced, integrator='midpoint', traindata=traindata_forced,
    epochs=30, batch_size=32,
)

print('\nTraining complete.')

In [ ]:
# Compare on the forced system
x0_f = [1.0, 0.0]
x_exact_f, *_ = forced_system.sample_trajectory(t_axis, x0=x0_f)
x_phnn_f, _ = phmodel_forced.simulate_trajectory(integrator=False, t_sample=t_axis, x0=x0_f)
x_base_f, _ = basemodel_forced.simulate_trajectory(integrator=False, t_sample=t_axis, x0=x0_f)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.plot(x_exact_f[:, 0], x_exact_f[:, 1], 'k--', linewidth=2, label='Exact')
ax.plot(x_phnn_f[:, 0], x_phnn_f[:, 1], 'b-', linewidth=1.5, label='PHNN')
ax.plot(x_base_f[:, 0], x_base_f[:, 1], 'r-', linewidth=1.5, label='Baseline')
ax.plot(x0_f[0], x0_f[1], 'ko', markersize=8)
ax.set_xlabel('Position (q)')
ax.set_ylabel('Momentum (p)')
ax.set_title('Forced System: Phase Portrait')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(t_axis, x_exact_f[:, 0], 'k--', linewidth=2, label='Exact')
ax.plot(t_axis, x_phnn_f[:, 0], 'b-', linewidth=1.5, label='PHNN')
ax.plot(t_axis, x_base_f[:, 0], 'r-', linewidth=1.5, label='Baseline')
ax.set_xlabel('Time')
ax.set_ylabel('Position (q)')
ax.set_title('Forced System: Position vs Time')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Damping recovered
learned_damping_f = phmodel_forced.R().detach().numpy()[1, 1]
print(f'\nDamping constant (forced system):')
print(f'  True:    {true_damping:.4f}')
print(f'  Learned: {learned_damping_f:.4f}')

## 8. Summary

| Property | PHNN | Standard NN |
|----------|------|-------------|
| Energy conservation | Guaranteed by architecture | Not enforced |
| Long-horizon stability | Physically plausible | Drifts to unphysical states |
| Dissipation | Learned, always non-negative | Unconstrained |
| External forces | Modular, can add/remove | Entangled in weights |
| Interpretability | Sub-networks map to physics | Black box |
| Parameter recovery | Recovers true damping | N/A |

The structural constraints cost nothing in training time but prevent entire classes of unphysical predictions. For further exploration, try the other example notebooks in `example_scripts/`:

- `phnn_ode_examples.ipynb` — mass-spring + tank systems with more configuration options
- `phnn_pde_examples.ipynb` — KdV, Cahn-Hilliard, and BBM equations
- `kdv_example.ipynb` — soliton propagation with PHNNs

---

*NAIC WP7 UC3 — Pseudo-Hamiltonian Neural Networks*  
*Contributors: Sølve Eidnes, Kjetil Olsen Lye (SINTEF Digital)*  
*NAIC Portal: https://orchestrator.naic.no*